In [0]:
# Instala os pacotes necessários para o projeto
%pip install python-dotenv azure-storage-file-datalake azure-identity pandas pyarrow pyodbc --quiet

dbutils.library.restartPython()

# Checa se existe um arquivo .env no ambiente
import os
from dotenv import load_dotenv, find_dotenv

dotenv_path = find_dotenv()

if dotenv_path:
    load_dotenv(dotenv_path, override=True)
    print(f"Variáveis encontradas: {dotenv_path} ")
else:
    print("Variáveis não encontradas.")

# Monta um dicionário reunindo as credenciais de acesso
adls_credentials = {
    "ADLS_CLIENT_ID": os.getenv("ADLS_CLIENT_ID"),
    "ADLS_TENANT_ID": os.getenv("ADLS_TENANT_ID"),
    "ADLS_CLIENT_SECRET": os.getenv("ADLS_CLIENT_SECRET"),
    "STORAGE_ACCOUNT_NAME": os.getenv("STORAGE_ACCOUNT_NAME"),
    "ADLS_CONTAINER": os.getenv("ADLS_CONTAINER", "raw")
}

jdbc_credentials = {
    "SQL_HOST": os.getenv("SQL_HOST"),
    "SQL_DATABASE": os.getenv("SQL_DATABASE"),
    "SQL_USERNAME": os.getenv("SQL_USERNAME"),
    "SQL_PASSWORD": os.getenv("SQL_PASSWORD")
}

print("\nStatus das Credenciais ADLS")

# Confere se cada credencial foi carregada corretamente
for k, v in adls_credentials.items():
    print(f"  {k}: {'[DEFINIDO]' if v else '[NÃO DEFINIDO]'}")

print("\nStatus das Credenciais SQL Server")

for k, v in jdbc_credentials.items():
    print(f"  {k}: {'[DEFINIDO]' if v else '[NÃO DEFINIDO]'}")


all_defined  = all(
    list(adls_credentials.values()) +
    list(jdbc_credentials.values())
)

if all_defined :
    print("\nVariáveis carregadas!")
else:
    print("\nExistem variáveis pendentes!")

# Configurações OAuth que serão usadas pelo Spark para autenticação
adls_options = {
    f"fs.azure.account.auth.type.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": adls_credentials["ADLS_CLIENT_ID"],
    f"fs.azure.account.oauth2.client.secret.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": adls_credentials["ADLS_CLIENT_SECRET"],
    f"fs.azure.account.oauth2.client.endpoint.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": f"https://login.microsoftonline.com/{adls_credentials["ADLS_TENANT_ID"]}/oauth2/token"
}

# Aponta o endereço da pasta batch-data dentro do Container
batch_data_path = (
    f"abfss://{adls_credentials["ADLS_CONTAINER"]}@{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net/"
    "batch-data"
)

# Busca e exibe os itens presentes no Container
files = (
    spark.read
    .format("binaryFile")
    .options(**adls_options)
    .load(batch_data_path)
    .select("path")
)

files.show(truncate=False)



Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Variáveis encontradas: /Workspace/Users/eng.stephanni@gmail.com/.env 

Status das Credenciais ADLS
  ADLS_CLIENT_ID: [DEFINIDO]
  ADLS_TENANT_ID: [DEFINIDO]
  ADLS_CLIENT_SECRET: [DEFINIDO]
  STORAGE_ACCOUNT_NAME: [DEFINIDO]
  ADLS_CONTAINER: [DEFINIDO]

Status das Credenciais SQL Server
  SQL_HOST: [DEFINIDO]
  SQL_DATABASE: [DEFINIDO]
  SQL_USERNAME: [DEFINIDO]
  SQL_PASSWORD: [DEFINIDO]

Variáveis carregadas!
+--------------------------------------------------------------------------------------------------+
|path                                                                                              |
+--------------------------------------------------------------------------------------------------+
|abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/physical_itens_venda_caixa.csv     |
|abfss://raw@internshipdatalake.dfs.core.windows.net/batc